In [ ]:
import time
start_time = time.time()

import gc
import platform
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sentence_transformers import CrossEncoder
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix


In [ ]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
model_name = "cross-encoder/ms-marco-MiniLM-L-6-v2"
batch_size = 64
threshold = 0.5

print(f"Platform: {platform.platform()}")
print(f"PyTorch version: {torch.__version__}")
print(f"Selected device: {device}")
print(f"Model: {model_name}")
print(f"Batch size: {batch_size}")
print(f"Threshold: {threshold}")


In [ ]:
dataset = load_dataset("glue", "mrpc", split="validation")
print(f"Validation examples: {len(dataset)}")

preview_df = dataset.select(range(min(5, len(dataset)))).to_pandas()[["sentence1", "sentence2", "label"]]
print(preview_df.to_string(index=False))


In [ ]:
model = CrossEncoder(model_name, device=device)
print(f"Loaded model: {model_name}")


In [ ]:
sentence_pairs = list(zip(dataset["sentence1"], dataset["sentence2"]))
true_labels = np.asarray(dataset["label"], dtype=np.int64)

inference_start = time.time()
scores = model.predict(sentence_pairs, batch_size=batch_size, show_progress_bar=True)
inference_seconds = time.time() - inference_start

scores = np.asarray(scores, dtype=np.float32)
predictions = (scores >= threshold).astype(np.int64)

examples_per_second = len(scores) / inference_seconds if inference_seconds > 0 else float("inf")

print(f"Completed inference for {len(predictions)} examples.")
print(f"Inference runtime (seconds): {inference_seconds:.2f}")
print(f"Examples per second: {examples_per_second:.2f}")
print(f"Score range: min={float(scores.min()):.4f}, max={float(scores.max()):.4f}")


In [ ]:
accuracy = accuracy_score(true_labels, predictions)
precision, recall, f1, _ = precision_recall_fscore_support(
    true_labels,
    predictions,
    average="binary",
    zero_division=0
)
cm = confusion_matrix(true_labels, predictions)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print("Confusion Matrix:")
print(cm)


In [ ]:
elapsed_seconds = time.time() - start_time

results_df = pd.DataFrame([
    {
        "model_name": model_name,
        "dataset": "glue/mrpc",
        "split": "validation",
        "num_examples": int(len(dataset)),
        "batch_size": int(batch_size),
        "threshold": float(threshold),
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "inference_seconds": float(inference_seconds),
        "total_runtime_seconds": float(elapsed_seconds),
        "examples_per_second": float(examples_per_second),
        "device": device
    }
])

print(results_df.to_string(index=False))


In [ ]:
examples_df = pd.DataFrame({
    "sentence1": dataset["sentence1"],
    "sentence2": dataset["sentence2"],
    "true_label": true_labels,
    "score": scores,
    "predicted_label": predictions
})

print(examples_df.head(10).to_string(index=False))

mismatches_df = examples_df.loc[examples_df["true_label"] != examples_df["predicted_label"]]
print(f"\nMismatches: {len(mismatches_df)}")
if len(mismatches_df) > 0:
    print(mismatches_df.head(10).to_string(index=False))

del sentence_pairs
gc.collect()

print(f"\nTotal runtime (seconds): {time.time() - start_time:.2f}")
